**使用指南：**

本脚本用于从消费者清洗表筛选重试对象，回查 bronze 原始消息后重新写入 bronze，并按需生成 topic 分批批次号。

！！！使用前，请确认环境配置（通过 get_env_config 读取）：
- silver_consumer_cleansed_database
- bronze_path_consumer
- config_database（仅用于步骤日志/分批日志）

1. 首次使用
- 先 Run all 一次，生成 widgets 输入框。
- 必填参数：
  - task_id：本次重试任务号（同时作为 slndc_batch_id 使用）
  - retry_condition_str：筛选条件（作用于 t_clean_consumer）
- 可选参数：
  - is_latest：True/False，默认 True；True 时按 business key 取最新一条
  - enable_batch_split：True/False，默认 False；是否启用按 topic 分批
  - max_topic_batch_count：整数字符串，默认 0；仅在 enable_batch_split=True 时生效

2. 重试候选筛选规则
- 来源表：{silver_consumer_cleansed_database}.t_clean_consumer
- business key：market_code, brand_code, source_system_code, consumer_id
- is_latest=True：每个 business key 按 srcc_sourcetimestamp desc、srcc_id desc 取最新
- is_latest=False：取条件命中的全部去重 srcc_id

3. bronze 回查与重建
- 通过 srcc_id = slndc_id 关联 bronze_path_consumer
- 取字段：slndc_kafka_topic, slndc_kafka_partition, slndc_kafka_offset,
  slndc_kafka_timestamp, slndc_kafka_key, slndc_payload
- 重新生成 slndc_id，并写入当前 task_id/slndc_batch_id

4. 分批规则（batch_number）
- 条件：enable_batch_split=True 且 max_topic_batch_count > 0
- 规则：按 slndc_kafka_topic 分区，按 slndc_kafka_timestamp 升序编号
- batch_number：{topic}_{slndc_batch_id}_{0001/0002/...}
- 不分批时固定：{topic}_{slndc_batch_id}_0001

5. 输出与写入
- 重试数据会 append 写入 bronze_path_consumer
- 分批开启时，会写入 {config_database}.t_topic_batch_log
- 脚本始终写入 t_task_step_log 记录步骤执行状态

6. retry_condition_str 示例
- _TASK_ID = "xxx" and SRCC_CONSUMERID = "xxx"
- SRCC_MRKT_CODE = "THA" and SRCC_SRCS_CODE = "CRM"

7. 注意事项
- retry_condition_str 不能为空，否则脚本会报错退出。
- max_topic_batch_count 建议大于 0 的正整数。
- task_id 建议全局唯一，避免与历史重试批次混淆。

In [0]:
%run ../00_common/data_utils

In [0]:
import time
from pyspark.sql.window import Window
from pyspark.sql import functions as F

def parse_bool(v):
    return str(v).strip().lower() == "true"

def parse_int(v, default_value=0):
    try:
        return int(str(v).strip())
    except Exception:
        return default_value

dbutils.widgets.text("task_id", "")
retry_task_id = dbutils.widgets.get("task_id")
# 取数的条件字符串
dbutils.widgets.text("retry_condition_str", "")
condition_str = dbutils.widgets.get("retry_condition_str")
# 是否拉取最新的数据(market_code, srcc_id, 默认true)
dbutils.widgets.text("is_latest", 'True')
is_latest = parse_bool(dbutils.widgets.get("is_latest"))
# 是否启用 topic 分批处理(True/False)
dbutils.widgets.text("enable_batch_split", 'False')
enable_batch_split = parse_bool(dbutils.widgets.get("enable_batch_split"))
# topic 每批最大处理数量(整型字符串)
dbutils.widgets.text("max_topic_batch_count", '0')
max_topic_batch_count = parse_int(dbutils.widgets.get("max_topic_batch_count"), 0)

print(f"retry_task_id: [{retry_task_id}]")
print(f"condition_str: [{condition_str}]")
print(f"is_latest: {is_latest}")
print(f"enable_batch_split: {enable_batch_split}")
print(f"max_topic_batch_count: {max_topic_batch_count}")

def find_retry_candidate_df(retry_task_id, condition_str, is_latest, param_bronze_path):
    # ----------------------------
    # 1. 获取当前时间（UTC）
    # ----------------------------
    exec_time = datetime.utcnow()

    # 2. 从Silver消费者清洗表中，筛选出满足条件的数据，构建base_df
    base_df = (
        spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_consumer")
        .where(condition_str)
        .select(
            F.col("srcc_mrkt_code").alias("market_code"),
            F.col("srcc_id"),
            F.col("srcc_brnd_code"),
            F.col("srcc_srcs_code"),
            F.col("srcc_consumerid"),
            F.col("srcc_sourcetimestamp")
        )
    )

    # 3. 根据is_latest参数，决定是取最新的数据还是全部数据
    if not is_latest:
        latest_df = base_df.select("market_code", "srcc_id").distinct()
    else:
        latest_df = (
            base_df
            .withColumn(
                "rank_num",
                F.row_number().over(
                    Window.partitionBy(
                        F.col("market_code"),
                        F.col("srcc_brnd_code"),
                        F.col("srcc_srcs_code"),
                        F.col("srcc_consumerid")
                    ).orderBy(
                        F.col("srcc_sourcetimestamp").desc_nulls_last(),
                        F.col("srcc_id").desc()
                    )
                )
            )
            .filter(F.col("rank_num") == 1)
            .select("market_code", "srcc_id")
            .distinct()
        )

    # 4. 根据条件过滤出的数据，关联bronze层数据，构建重试的源数据
    bronze_df = spark.read.format("delta").load(param_bronze_path)
    retry_source_df = (
        latest_df.alias("l")
        .join(
            bronze_df.alias("b"),
            F.col("l.srcc_id").cast("string") == F.col("b.slndc_id").cast("string"),
            "inner"
        )
        .select(
            F.expr("uuid()").alias("slndc_id"),
            F.lit(retry_task_id).alias("task_id"),
            F.lit(retry_task_id).alias("slndc_batch_id"), # 暂时用task_id作为batch_id，后续可以根据需要调整
            F.lit(exec_time).cast("timestamp").alias("slndc_batch_dt"),
            F.col("b.slndc_kafka_topic"),
            F.col("b.slndc_kafka_partition"),
            F.col("b.slndc_kafka_offset"),
            F.col("b.slndc_kafka_timestamp"),
            F.col("b.slndc_kafka_key"),
            F.col("b.slndc_payload"),
            F.current_timestamp().alias("slndc_creation_dt"),
            F.lit("").alias("slndc_creationuid"),
            F.current_timestamp().alias("slndc_update_dt"),
            F.lit("").alias("slndc_updateuid")
        )
        .distinct()
    )
    return retry_source_df

def generate_main_retry_data(retry_task_id,
                             condition_str, 
                             is_latest, 
                             param_bronze_path,
                             enable_batch_split, 
                             max_topic_batch_count):
    
    if not condition_str or condition_str.strip() == '':
        raise Exception("condition_str must not be blank.")
    
    log_table_batch = f"{get_env_config('config_database')}.t_topic_batch_log"

    # 1.查询并构建要重试的bronze源数据
    candidate_df = find_retry_candidate_df(retry_task_id, condition_str, is_latest, param_bronze_path)

    # 2.topic分批处理(重试数据拆分批次)，并生成batch_number批次号
    if enable_batch_split and max_topic_batch_count > 0:
        window_spec = Window.partitionBy("slndc_kafka_topic").orderBy(F.col("slndc_kafka_timestamp").asc())
        bronze_with_retry = (
            candidate_df
            .withColumn("row_num", F.row_number().over(window_spec))
            .withColumn("batch_seq", F.ceil(F.col("row_num") / max_topic_batch_count).cast("int"))
            .withColumn("batch_seq_str", F.format_string("%04d", F.col("batch_seq")))
            .withColumn(
                "batch_number",
                F.concat_ws("_", F.col("slndc_kafka_topic"), F.col("slndc_batch_id"), F.col("batch_seq_str"))
            )
            .drop("row_num", "batch_seq", "batch_seq_str")
        )
    else:
        bronze_with_retry = candidate_df.withColumn(
            "batch_number",
            F.concat_ws("_", F.col("slndc_kafka_topic"), F.col("slndc_batch_id"), F.lit("0001"))
        )

    bronze_with_retry.cache()
    try:
        retry_bronze_count = bronze_with_retry.count()
        if retry_bronze_count > 0:
            # 3.写入Bronze Delta表
            (
                bronze_with_retry
                .write
                .format("delta")
                .mode("append")
                .option("mergeSchema", "true")
                .option("path", param_bronze_path)
                .save()
            )
            print(f"[retry-main] Appended {retry_bronze_count} records to bronze table at {param_bronze_path}")

            # 4.写入Topic批次日志表
            if enable_batch_split and max_topic_batch_count > 0:
                write_batch_log(bronze_with_retry, log_table_batch)
    finally:
        bronze_with_retry.unpersist()

In [0]:
with StepLogger("generate_main_retry_data", "10", "consumerlist", task_id=retry_task_id) as logger:
    # main
    generate_main_retry_data(
        retry_task_id=retry_task_id,
        condition_str=condition_str,
        is_latest=is_latest,
        param_bronze_path=get_env_config("bronze_path_consumer"),
        enable_batch_split=enable_batch_split,
        max_topic_batch_count=max_topic_batch_count
    )